# NarraBERT Likert model

[`CLS-Lab/narrative-likert-roberta`](https://huggingface.co/CLS-Lab/narrative-likert-roberta) scores a given
passage on nine narrative dimensions, each a continuous 1–5 regression output.

In [4]:
import json

import pandas as pd
import torch
from torch import nn
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoTokenizer

REPO_ID = "CLS-Lab/narrative-likert-roberta"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


class NarrativeLikertRoBERTa(nn.Module):
    """RoBERTa backbone + one linear regression head per dimension, on [CLS]."""

    def __init__(self, model_name, n_dims):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.heads = nn.ModuleList(
            [nn.Linear(self.backbone.config.hidden_size, 1) for _ in range(n_dims)]
        )

    def forward(self, input_ids, attention_mask):
        cls = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return torch.cat([h(cls) for h in self.heads], dim=-1)


# The checkpoint is a raw state_dict, so AutoModel.from_pretrained(REPO_ID) will not work.
config = json.load(open(hf_hub_download(REPO_ID, "config.json")))
MAX_LEN, DIMS = config["max_len"], config["dims"]

tokenizer = AutoTokenizer.from_pretrained(REPO_ID, subfolder="tokenizer")
model = NarrativeLikertRoBERTa(config["model_name"], len(DIMS))
# Load on CPU first; loading straight to mps raises "Unaligned blit request".
model.load_state_dict(
    torch.load(hf_hub_download(REPO_ID, "model.pt"), map_location="cpu", weights_only=True)
)
model.to(device).eval()

print(f"{device}, max_len={MAX_LEN}, dims={DIMS}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


mps, max_len=200, dims=['focalization', 'emotion', 'cognition', 'change_of_state', 'conflict', 'concreteness', 'temporal_grounding', 'spatial_grounding', 'sensory']


## Scoring
| Group | Dimensions |
|---|---|
| Agency | `focalization`, `emotion`, `cognition`, `change_of_state`, `conflict` |
| Setting | `concreteness`, `temporal_grounding`, `spatial_grounding`, `sensory` |

Below we test two very simple, contrived examples to show how you can apply NarraBERT to your own text.

In [7]:
@torch.no_grad()
def score(texts, batch_size=32):
    """Score passages. Returns a DataFrame with one column per dimension."""
    rows = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i + batch_size],
            max_length=MAX_LEN, padding="max_length", truncation=True, return_tensors="pt",
        )
        out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        rows.extend(out.clamp(1.0, 5.0).cpu().float().tolist())  # heads are unbounded; the scale is not
    return pd.DataFrame(rows, columns=DIMS)


score([
    "I walked into the interview room and immediately noticed the panel was twice the size I "
    "expected. My hands started shaking. I sat down, took a breath, and got through the first question.",
    "The library is open from 9am to 5pm on weekdays. Renewals may be processed online or at the "
    "front desk. Late fees accrue at ten cents per day per item. On February 1st 2028, the library will be closed for renovations.",
]).round(2)

,focalization,emotion,cognition,change_of_state,conflict,concreteness,temporal_grounding,spatial_grounding,sensory
0,4.45,2.74,2.65,2.16,2.27,2.98,1.16,3.10,3.0
1,1.03,1.00,1.06,1.93,1.00,1.85,4.42,2.69,1.0


## Evaluating against human gold labels

[`CLS-Lab/narrative-gold-annotations`](https://huggingface.co/datasets/CLS-Lab/narrative-gold-annotations)
holds gold human ratings for 400 passages, split into an `agency` and a `setting` config over the
same `safe_instance_id`s. Columns are named `{group}_{dimension}_gold`.

To show an example, we just apply NarraBERT to these gold labels. You can recompute the NarraBERT agreement scores from here, or upload your own data.

In [6]:
GROUP = dict.fromkeys(DIMS[:5], "agency") | dict.fromkeys(DIMS[5:], "setting")

agency = load_dataset("CLS-Lab/narrative-gold-annotations", "agency", split="train").to_pandas()
setting = load_dataset("CLS-Lab/narrative-gold-annotations", "setting", split="train").to_pandas()
gold = agency.merge(
    setting.drop(columns=["folder", "dolma_source", "sampled_text"]), on="safe_instance_id"
)

pred = score(gold["sampled_text"].tolist())

pd.DataFrame(
    [
        {
            "n": int(gold[f"{GROUP[d]}_{d}_gold"].notna().sum()),
            "gold_mean": gold[f"{GROUP[d]}_{d}_gold"].mean(),
            "pred_mean": pred[d].mean(),
            "MAE": (pred[d] - gold[f"{GROUP[d]}_{d}_gold"]).abs().mean(),
            "pearson_r": pred[d].corr(gold[f"{GROUP[d]}_{d}_gold"]),
        }
        for d in DIMS
    ],
    index=DIMS,
).round(3)

,n,gold_mean,pred_mean,MAE,pearson_r
focalization,400,2.418,2.078,0.553,0.835
emotion,400,1.858,1.682,0.451,0.795
cognition,400,2.198,2.167,0.585,0.753
change_of_state,400,2.385,2.147,0.712,0.678
conflict,400,2.070,1.928,0.582,0.769
concreteness,400,2.480,2.387,0.561,0.681
temporal_grounding,400,2.152,2.413,0.556,0.817
spatial_grounding,400,2.115,2.606,0.624,0.850
sensory,399,1.561,1.557,0.396,0.721
